In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [2]:
import bs4
from dotenv import load_dotenv
import langchainhub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

C:\Users\HP\AppData\Local\Temp\ipykernel_29848\2219484041.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
# later execute it to load the apis from .env
# load_dotenv()

In [3]:
os.environ["USER_AGENT"] = "MyFreeRAGApp/1.0"

In [4]:
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = <your-api-key>

SyntaxError: invalid syntax (4267060190.py, line 3)

In [6]:
os.environ['OPENAI_API_KEY'] = <your-api-key>

In [7]:
os.environ["GOOGLE_API_KEY"] = <your-api-key>

### Overview

In [10]:
#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (Free Local Hugging Face Model)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Index Chunks into Vector Store
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt

# hub_client = langchainhub.Client()
# prompt = hub_client.pull("rlm/rag-prompt")

# or use the below to avoid network call
from langchain_core.prompts import ChatPromptTemplate

# Recreate the exact 'rlm/rag-prompt' locally
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n"
    "Context: {context}\n"
    "Answer:"
)

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0
)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7. Execute
response = rag_chain.invoke("What is Task Decomposition?")
print(response)

Task decomposition is the process of breaking down a complex task into smaller, simpler, and more manageable steps or subgoals. It can be accomplished by a Large Language Model (LLM) using simple prompting, task-specific instructions, or through human inputs. Techniques like Chain of Thought (CoT) utilize task decomposition to help models plan ahead and solve hard tasks step-by-step.


### Indexing